**Ноутбук 02 — вимір валют**

In [1]:
!pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

In [8]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "project-nbu"   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-nbu


**Завдання 3.1.** Прочитайте з nbu_raw.raw_rates колонки ingested_at, business_date, payload

In [15]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = "project-nbu"   
LOCATION   = "EU"

creds = None

# Явно читаем переменную GCP_SA_KEY, которую вы создали в настройках Datalore
if os.environ.get("GCP_SA_KEY"):
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        # Вот этот адрес гарантирует полный доступ к BigQuery и решение всех ошибок:
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
else:
    raise Exception("Ошибка: Переменная GCP_SA_KEY не найдена в настройках Environment!")

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("Проєкт підключено успішно до Ноутбука 02:", client.project)

Проєкт підключено успішно до Ноутбука 02: project-nbu


In [16]:
# --- ЗАВДАННЯ 3.1: Читання обраних колонок із шару Bronze ---

query_bronze = f"""
SELECT ingested_at, business_date, payload
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""

# Теперь этот метод сработает идеально, так как авторизация правильная
df_bronze_rates = client.query(query_bronze).to_dataframe()

print(f"✅ Данные успешно загружены! Найдено строк: {len(df_bronze_rates)}")
print(df_bronze_rates.head(2))

✅ Данные успешно загружены! Найдено строк: 90
                       ingested_at business_date  \
0 2026-08-24 13:37:47.363434+00:00    2026-08-25   
1 2026-08-24 13:37:47.363434+00:00    2026-08-25   

                                             payload  
0  {"cc": "DZD", "exchangedate": "25.08.2026", "r...  
1  {"cc": "AUD", "exchangedate": "25.08.2026", "r...  


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 3.2.** Розгорніть JSON-текст із payload у колонки

In [17]:
# --- ЗАВДАННЯ 3.2: Развертывание JSON-текста из payload в колонки ---
import json

# Разворачиваем строки JSON в полноценные колонки DataFrame
parsed = pd.json_normalize(df_bronze_rates["payload"].map(json.loads))

# Проверка: выводим список получившихся колонок и первые 3 строки
print("Развернутые колонки из payload:", list(parsed.columns))
print("\nПервые 3 строки развернутых данных:")
print(parsed.head(3))

Развернутые колонки из payload: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt']

Первые 3 строки развернутых данных:
    cc exchangedate  r030      rate special                   txt
0  DZD   25.08.2026    12   0.33615    None      Алжирський динар
1  AUD   25.08.2026    36  32.03210    None  Австралійський долар
2  BDT   25.08.2026    50   0.36460    None                  Така


**Завдання 3.3.** Нормалізуйте значення: cc — прибрати пробіли й перевести у верхній регістр, txt — прибрати пробіли з країв, r030 — перевести у цілий тип Int64

In [18]:
# --- ЗАВДАННЯ 3.3: Нормализация значений полей валют ---

# 1. Очищаем cc (код валюты): убираем пробелы и переводим в верхний регистр
parsed["cc"] = parsed["cc"].str.strip().str.upper()

# 2. Очищаем txt (название): убираем пробелы с краев
parsed["txt"] = parsed["txt"].str.strip()

# 3. Приводим r030 (цифровой код) к целому типу Int64 (с поддержкой NaN, если они будут)
parsed["r030"] = parsed["r030"].astype("Int64")

# Проверка: выводим типы данных и пример очищенных строк
print("--- Проверка типов данных ---")
print(parsed[["cc", "txt", "r030"]].dtypes)

print("\n--- Пример очищенных данных (первые 3 строки) ---")
print(parsed[["cc", "txt", "r030"]].head(3))

--- Проверка типов данных ---
cc      object
txt     object
r030     Int64
dtype: object

--- Пример очищенных данных (первые 3 строки) ---
    cc                   txt  r030
0  DZD      Алжирський динар    12
1  AUD  Австралійський долар    36
2  BDT                  Така    50


**Завдання 3.4.** Залиште по одному рядку на валюту, узявши найсвіжіший запис за ingested_at. Назва валюти в джерелі може змінитися, тому важливо брати саме останню версію, а не першу.

In [19]:
# --- ЗАВДАННЯ 3.4: Дедупликация и выбор самых свежих записей валют ---

# 1. Добавляем колонку ingested_at из исходного бронзового датафрейма
parsed["ingested_at"] = df_bronze_rates["ingested_at"]

# 2. Сортируем по времени и оставляем только последнюю (самую свежую) запись для каждой валюты
dim_currency = (
    parsed.sort_values("ingested_at")
    .drop_duplicates(subset="cc", keep="last")
    .copy()
)

# Проверка: количество уникальных валют в справочнике (обычно около 60)
print(f"Количество строк в итоговом измерении валют: {len(dim_currency)}")

Количество строк в итоговом измерении валют: 45


**Завдання 3.5.** Відсортуйте за currency_code і додайте першою колонкою сурогатний ключ currency_key — послідовні числа від 1.

In [20]:
# --- ЗАВДАННЯ 3.5: Сортування та додавання сурогатного ключа ---

# 1. Сортуємо за кодом валюти та скидаємо індекси на нові (від 0)
dim_currency = dim_currency.sort_values("cc").reset_index(drop=True)

# 2. Додаємо першою колонкою сурогатний ключ currency_key (послідовні числа від 1)
dim_currency.insert(0, "currency_key", range(1, len(dim_currency) + 1))

# Перевірка: виводимо розмір та перші 5 рядків готового виміру
print(f"Розмір таблиці виміру: {dim_currency.shape} (рядків, колонок)")
print("\nФрагмент готової таблиці виміру dim_currency:")
print(dim_currency[["currency_key", "cc", "txt", "r030"]].head())

Розмір таблиці виміру: (45, 8) (рядків, колонок)

Фрагмент готової таблиці виміру dim_currency:
   currency_key   cc                     txt  r030
0             1  AED              Дирхам ОАЕ   784
1             2  AUD    Австралійський долар    36
2             3  AZN  Азербайджанський манат   944
3             4  BDT                    Така    50
4             5  CAD        Канадський долар   124


**Завдання 3.6.** Додайте рядок Unknown із ключем -1: currency_code = N/A, currency_name = Unknown, r030 = -1. Він знадобиться в завданні 6, якщо валюта із фактів не знайдеться у вимірі.

In [21]:
# --- ЗАВДАННЯ 3.6: Додавання технічного рядка Unknown з ключем -1 ---

# 1. Створюємо словник із дефолтними значеннями для невідомої валюти
unknown_row = {
    "currency_key": -1,
    "cc": "N/A",        # літерний код валюти
    "txt": "Unknown",   # назва валюти
    "r030": -1          # цифровий код валюти
}

# 2. Перетворюємо словник у DataFrame з одним рядком
df_unknown = pd.DataFrame([unknown_row])

# 3. Об'єднуємо рядок Unknown з основним виміром dim_currency
dim_currency = pd.concat([df_unknown, dim_currency], ignore_index=True)

# Перевірка: виводимо перші 3 рядки, щоб побачити Unknown на початку
print("--- Фрагмент таблиці з рядком Unknown на початку ---")
print(dim_currency[["currency_key", "cc", "txt", "r030"]].head(3))

--- Фрагмент таблиці з рядком Unknown на початку ---
   currency_key   cc                   txt  r030
0            -1  N/A               Unknown    -1
1             1  AED            Дирхам ОАЕ   784
2             2  AUD  Австралійський долар    36


**Завдання 3.7.** Запишіть результат у nbu_dwh.dim_currency у режимі WRITE_TRUNCATE.
Отримати на виході: currency_key, currency_code, currency_name, r030

In [22]:
# --- ЗАВДАННЯ 3.7: Фінальний відбір колонок та запис у шар Gold ---

TABLE_ID = f"{PROJECT_ID}.nbu_dwh.dim_currency"

# 1. Перейменовуємо наші поточні колонки на цільові назви
dim_currency_final = dim_currency.rename(columns={
    "cc": "currency_code",
    "txt": "currency_name"
})

# 2. Відбираємо тільки необхідні колонки у заданому порядку
target_columns = ["currency_key", "currency_code", "currency_name", "r030"]
dim_currency_final = dim_currency_final[target_columns]

# 3. Налаштовуємо конфігурацію запису: повний перезапис таблиці (WRITE_TRUNCATE)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# 4. Записуємо DataFrame у BigQuery
job = client.load_table_from_dataframe(dim_currency_final, TABLE_ID, job_config=job_config)
job.result()  # Очікуємо завершення завантаження

print(f"✅ Вимір успішно збережено в таблицю {TABLE_ID}!")
print(f"Кількість рядків у сховищі: {len(dim_currency_final)}")
print("\nПерші 3 рядки на виході:")
print(dim_currency_final.head(3))

✅ Вимір успішно збережено в таблицю project-nbu.nbu_dwh.dim_currency!
Кількість рядків у сховищі: 46

Перші 3 рядки на виході:
   currency_key currency_code         currency_name  r030
0            -1           N/A               Unknown    -1
1             1           AED            Дирхам ОАЕ   784
2             2           AUD  Австралійський долар    36


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


**Завдання 3.8.** Перевірте трьома рядками й виведіть результат: currency_code унікальний; у таблиці є рядок із currency_key = -1; кількість рядків дорівнює кількості унікальних валют у bronze плюс один.

In [23]:
# --- ЗАВДАННЯ 3.8: Фінальна валідація виміру dim_currency ---

# 1. Перевірка: чи є унікальним стовпчик currency_code (ігноруємо технічний 'N/A')
is_unique = dim_currency_final[dim_currency_final["currency_code"] != "N/A"]["currency_code"].is_unique

# 2. Перевірка: чи є в таблиці рядок із технічним ключем currency_key = -1
has_unknown = (dim_currency_final["currency_key"] == -1).any()

# 3. Перевірка: чи дорівнює кількість рядків кількості унікальних валют у bronze плюс один
expected_rows_cnt = parsed["cc"].str.strip().str.upper().nunique() + 1
count_matches = len(dim_currency_final) == expected_rows_cnt

# Виведення результатів перевірки
print(is_unique)
print(has_unknown)
print(count_matches)

True
True
True
